# emg2qwerty Checkpoint Test Runner

极简版：只加载一个已经训练好的 checkpoint，在当前数据集上跑 `train=False` 的 `val`/`test`。

In [2]:
from pathlib import Path
import sys
import subprocess

REPO_ROOT = Path.cwd().resolve()
DATA_DIR = REPO_ROOT / "data" / "subject_89335547"
CKPT_PATH = REPO_ROOT / "models" / "generic.ckpt" 

def to_posix(p: Path) -> str:
    return str(p).replace("\\", "/")

patch_and_run = r'''\
import torch, runpy
_old = torch.load

def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _old(*args, **kwargs)

torch.load = _patched_load
runpy.run_module("emg2qwerty.train", run_name="__main__")
'''.strip()

cmd = [
    sys.executable,
    "-c",
    patch_and_run,
    f"dataset.root={to_posix(DATA_DIR)}",
    "train=False",
    f"checkpoint={to_posix(CKPT_PATH)}",
]

print("将要运行的命令:\n")
print(" ".join(cmd))

subprocess.run(cmd, cwd=REPO_ROOT, check=True)


将要运行的命令:

c:\Users\15855\.conda\envs\emg2qwerty\python.exe -c \
import torch, runpy
_old = torch.load

def _patched_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _old(*args, **kwargs)

torch.load = _patched_load
runpy.run_module("emg2qwerty.train", run_name="__main__") dataset.root=C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/data/subject_89335547 train=False checkpoint=C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/models/generic.ckpt


CompletedProcess(args=['c:\\Users\\15855\\.conda\\envs\\emg2qwerty\\python.exe', '-c', '\\\nimport torch, runpy\n_old = torch.load\n\ndef _patched_load(*args, **kwargs):\n    kwargs.setdefault("weights_only", False)\n    return _old(*args, **kwargs)\n\ntorch.load = _patched_load\nrunpy.run_module("emg2qwerty.train", run_name="__main__")', 'dataset.root=C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/data/subject_89335547', 'train=False', 'checkpoint=C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/models/generic.ckpt'], returncode=0)

In [3]:
from pathlib import Path

# 简单辅助：打印最新一次 run 的日志最后若干行，方便看 val/test 结果
logs_root = Path("logs/2026-03-10")  # 如需以后跑别的日期，可以改成 Path("logs") 再选最新子目录
latest_run_dir = max(logs_root.iterdir(), key=lambda p: p.name)
log_path = latest_run_dir / "emg2qwerty.log"
print("Using log:", log_path)

with log_path.open("r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines[-60:]:  # 打印最后 60 行，里面一般会有 val/test metrics
    print(line.rstrip("\n"))


Using log: logs\2026-03-10\03-44-58\emg2qwerty.log
  - ${specaug}
  val:
  - ${to_tensor}
  - ${logspec}
  test: ${transforms.val}
module:
  _target_: emg2qwerty.lightning.TDSConvCTCModule
  in_features: 528
  mlp_features:
  - 384
  block_channels:
  - 24
  - 24
  - 24
  - 24
  kernel_width: 32
datamodule:
  _target_: emg2qwerty.lightning.WindowedEMGDataModule
  window_length: 8000
  padding:
  - 1800
  - 200
optimizer:
  _target_: torch.optim.Adam
  lr: 0.001
lr_scheduler:
  scheduler:
    _target_: pl_bolts.optimizers.lr_scheduler.LinearWarmupCosineAnnealingLR
    warmup_epochs: 10
    max_epochs: ${trainer.max_epochs}
    warmup_start_lr: 1.0e-08
    eta_min: 1.0e-06
  interval: epoch
decoder:
  _target_: emg2qwerty.decoder.CTCGreedyDecoder
seed: 1501
batch_size: 32
num_workers: 4
train: false
checkpoint: C:/Users/15855/OneDrive/UCLA/course/247A/Project/emg2qwerty/models/generic.ckpt
monitor_metric: val/CER
monitor_mode: min
trainer:
  accelerator: gpu
  devices: 1
  num_nodes: 1
 